# IR Practical 3: Bayesian Network for Heart Disease
**Aim:** Create a Discrete Bayesian Network, learn conditional probability distributions (CPDs), validate the model, and perform inference using Variable Elimination.

The notebook follows the model structure shown in the supplied PDF. It attempts the PDF's dataset URL first and includes a fallback dataset source in case that link is unavailable.

In [ ]:
%pip -q install -U pgmpy pandas

In [ ]:
import pandas as pd
import pgmpy
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

print("pgmpy version:", pgmpy.__version__)

In [ ]:
# Dataset URL shown in the PDF (the filename in the printout is partially clipped).
source_url = "https://gist.githubusercontent.com/prajwalmani/e4d76fa6f32adc53949c23db1ed037f2/raw/data7_heart.csv"

try:
    heartDisease = pd.read_csv(source_url)
    print("Dataset loaded from the practical's source URL.")
except Exception as exc:
    print("Could not load the practical URL:", exc)
    fallback_url = "https://raw.githubusercontent.com/plotly/datasets/master/heart.csv"
    heartDisease = pd.read_csv(fallback_url)
    print("Loaded fallback heart dataset. Check its columns before proceeding.")

print("\\nFirst 5 records:")
display(heartDisease.head())

In [ ]:
print("Dataset Shape:")
print(heartDisease.shape)

print("\\nDataset Columns:")
print(heartDisease.columns.tolist())

print("\\nMissing Values:")
print(heartDisease.isnull().sum())

print("\\nDataset Information:")
heartDisease.info()

In [ ]:
# Standardize common column-name variations and select the variables used in the PDF.
heartDisease.columns = [str(c).strip().lower().replace(" ", "_") for c in heartDisease.columns]

# The practical uses these exact variables.
required_columns = ["age", "sex", "cp", "exang", "restecg", "chol", "heartdisease"]

# Some common heart-disease datasets name the target 'target' or 'num'.
if "heartdisease" not in heartDisease.columns:
    if "target" in heartDisease.columns:
        heartDisease = heartDisease.rename(columns={"target": "heartdisease"})
    elif "num" in heartDisease.columns:
        heartDisease = heartDisease.rename(columns={"num": "heartdisease"})

missing_columns = [c for c in required_columns if c not in heartDisease.columns]
if missing_columns:
    raise ValueError(
        f"The loaded dataset does not contain required practical columns: {missing_columns}. "
        f"Available columns: {heartDisease.columns.tolist()}"
    )

heartDisease = heartDisease[required_columns].copy()

# Bayesian-network variables must be discrete. Convert values to numeric,
# remove rows with invalid/missing values, and use integer state labels.
for col in required_columns:
    heartDisease[col] = pd.to_numeric(heartDisease[col], errors="coerce")
heartDisease = heartDisease.dropna().copy()
for col in required_columns:
    heartDisease[col] = heartDisease[col].astype(int)

print("Data prepared for Bayesian Network:")
display(heartDisease.head())
print("Prepared shape:", heartDisease.shape)

In [ ]:
model = DiscreteBayesianNetwork([
    ("age", "heartdisease"),
    ("sex", "heartdisease"),
    ("exang", "heartdisease"),
    ("cp", "heartdisease"),
    ("heartdisease", "restecg"),
    ("heartdisease", "chol")
])

print("Bayesian Network created successfully!")

In [ ]:
print("Nodes:")
print(list(model.nodes()))

print("\\nEdges:")
print(list(model.edges()))

In [ ]:
# Learn the CPDs. Maximum-likelihood estimation is the estimator used in the PDF.
# pgmpy versions differ in whether the estimator is passed explicitly to fit().
try:
    from pgmpy.parameter_estimator import DiscreteMLE
    estimator = DiscreteMLE()
    model.fit(heartDisease, estimator=estimator)
except (ImportError, TypeError):
    model.fit(heartDisease)

print("CPDs learned successfully!")

In [ ]:
print("Learned Conditional Probability Distributions:\\n")
for cpd in model.get_cpds():
    print(cpd)
    print("=" * 70)

In [ ]:
print("Model Check:")
print(model.check_model())

In [ ]:
inference = VariableElimination(model)
print("Variable Elimination inference created successfully!")

In [ ]:
print("Probability of Heart Disease given cp = 2:\\n")

# This is the evidence/query used in the supplied practical.
if 2 not in set(heartDisease["cp"].unique()):
    print("The loaded dataset does not contain cp = 2; available cp states:",
          sorted(heartDisease["cp"].unique()))
else:
    q2 = inference.query(
        variables=["heartdisease"],
        evidence={"cp": 2}
    )
    print(q2)